# EDA — Synthetic Access-Log Dataset
Quick look at class balance, per-entity-type volume, and the injected attack campaigns before building features/models.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('../data/labeled/access_logs_labeled.csv', parse_dates=['timestamp'])
df.shape

In [ ]:
# Class balance — this is the imbalance the detector has to handle
df['label'].value_counts()

In [ ]:
anomaly_rate = (df['label'].str.startswith('anomaly_')).mean()
print(f'Hard anomaly rate: {anomaly_rate:.2%}')
edge_case_rate = (df['label'] == 'insider_drift_edge_case').mean()
print(f'Ambiguous edge-case rate: {edge_case_rate:.2%}')

In [ ]:
# Sessions per day, normal vs anomalous — sanity-check the time distribution
daily = df.assign(date=df['timestamp'].dt.date, is_anomaly=df['label'].str.startswith('anomaly_'))
daily.groupby(['date', 'is_anomaly']).size().unstack(fill_value=0).plot(figsize=(12, 4))
plt.title('Sessions per day')
plt.show()

In [ ]:
# Volume by entity_type
df.groupby('entity_type')['label'].value_counts(normalize=False).unstack(fill_value=0)

In [ ]:
# Attack campaign sizes (rows per attack_group_id) — informs sequence_length in config.yaml
df[df['attack_group_id'] != ''].groupby(['label', 'attack_group_id']).size().groupby('label').describe()